# Data Temporality: Temporal Post-processing for Faster R-CNN

Faster R-CNN processes each frame separately, so we test whether using information from neighboring frames can improve detection without retraining the model.

The method includes four stages:

- Camera motion: Estimating camera movement.

- Tracking: Tracking objects across frames.

- Rescoring and suppression: Adjusting detection confidence scores.

- Gap recovery: Recovering missed detections.

In [ ]:
import base64
import io
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from PIL import Image
from IPython.display import display

import sys
sys.path.insert(0, '../shared')
import experiment_utils
import temporal_postprocessing
import tta_wbf

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

In [ ]:
EXPERIMENT_DIR = experiment_utils.HERE
CACHE = experiment_utils.CACHE
FOLDS = experiment_utils.FOLDS
PLOTS_DIR = EXPERIMENT_DIR / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)
pio.renderers.default = 'plotly_mimetype+notebook_connected'

def save_plot(fig, name):
    fig.update_layout(template='plotly_white', font=dict(size=13))
    fig.write_html(PLOTS_DIR / f'{name}.html', include_plotlyjs='directory', div_id=name)
    fig.show()

In [ ]:
experiment_utils.build_caches(log=lambda message: print(time.strftime('%H:%M:%S'), message))

## Data and starting predictions

In [ ]:
frame_sets = {}
for set_name in ['tune', 'val', 'test']:
    frames = experiment_utils.frames_meta(set_name)
    per_video = frames.groupby('video').frame.agg(['count', 'min', 'max'])
    assert ((per_video['max'] - per_video['min'] + 1) == per_video['count']).all()
    assert all((np.diff(video.frame.values) == 1).all() for _, video in frames.groupby('video', sort=False))
    frame_sets[set_name] = frames
    resolutions = sorted(set(zip(frames.width, frames.height)))
    print(f"{set_name:5s}: {len(frames):5d} frames in {frames.video.nunique():2d} videos, "
          f"{per_video['count'].min()}-{per_video['count'].max()} frames per video | resolutions {resolutions}")

tune :  4458 frames in 56 videos, 58-80 frames per video | resolutions [(1344, 756), (1360, 765), (1904, 1071), (1920, 1080), (2688, 1512), (2720, 1530)]


val  :  2846 frames in  7 videos, 184-978 frames per video | resolutions [(1344, 756), (1904, 1071), (2688, 1512), (2720, 1530), (3840, 2160)]
test :  3811 frames in 48 videos, 35-148 frames per video | resolutions [(680, 382), (960, 540), (1344, 756), (1360, 764), (1904, 1072), (1920, 1080)]


In [ ]:
METRIC_NAMES = ['mAP', 'AP50', 'AP75', 'APs', 'APm', 'APl', 'AR']
saved_runs = experiment_utils.ROOT / 'faster_rcnn/official_validation/runs'
saved_metrics = {fold: json.loads((saved_runs / f'tta_wbf_fold{fold}.json').read_text())['row'] for fold in FOLDS}

val_frames, _ = experiment_utils.load_preds(CACHE / 'val_fold0.npz')
val_image_ids = val_frames.image_id.tolist()
val_gt = temporal_postprocessing.coco_gt(experiment_utils.annotations_for('val'), val_image_ids)
for fold in FOLDS:
    fold_predictions = experiment_utils.load_preds(CACHE / f'val_fold{fold}.npz')[1]
    metrics = temporal_postprocessing.coco_metrics(val_gt, val_image_ids, fold_predictions)['all']
    assert max(abs(metrics[name] - saved_metrics[fold][name]) for name in METRIC_NAMES) < 1e-12
print('single-checkpoint TTA + WBF on official validation reproduces the saved metrics exactly (5 checkpoints)')

single-checkpoint TTA + WBF on official validation reproduces the saved metrics exactly (5 checkpoints)


In [ ]:
test_frames, test_predictions = experiment_utils.load_preds(CACHE / 'test_ens.npz')
submitted_file = experiment_utils.ROOT / 'kaggle_submissions/faster_rcnn_tta_wbf_submission.csv'
submitted = pd.read_csv(submitted_file, dtype=str, keep_default_na=False)
rebuilt = [tta_wbf.prediction_string(boxes, scores, labels, height, width)
           for (boxes, scores, labels), height, width in zip(test_predictions, test_frames.height, test_frames.width)]
assert submitted.image_id.tolist() == test_frames.image_id.tolist()
assert submitted.PredictionString.tolist() == rebuilt
print('the 5-checkpoint ensemble rebuilt here equals the submitted 0.2587 CSV exactly (3811 rows)')

the 5-checkpoint ensemble rebuilt here equals the submitted 0.2587 CSV exactly (3811 rows)


## Camera motion

In [ ]:
camera_motion = {set_name: experiment_utils.motion_for(set_name) for set_name in frame_sets}

In [ ]:
rows = []
for set_name, (transforms, inliers, seconds) in camera_motion.items():
    first_in_video = frame_sets[set_name].groupby('video', sort=False).cumcount().values == 0
    pairs = ~first_in_video
    shift = np.hypot(transforms[pairs, 0, 2], transforms[pairs, 1, 2])
    zoom = np.sqrt(np.abs(np.linalg.det(transforms[pairs, :2, :2])))
    rotation = np.degrees(np.arctan2(transforms[pairs, 1, 0], transforms[pairs, 0, 0]))
    rows.append(dict(set=set_name, frame_pairs=int(pairs.sum()), median_shift_px=np.median(shift),
                     p90_shift_px=np.percentile(shift, 90), median_abs_rotation_deg=np.median(np.abs(rotation)),
                     median_abs_zoom=np.median(np.abs(zoom - 1)), median_inliers=int(np.median(inliers[pairs])),
                     failed_pairs=int((inliers[pairs] == 0).sum()), ms_per_frame=1000 * seconds.mean()))
display(pd.DataFrame(rows).round(3))

,set,frame_pairs,median_shift_px,p90_shift_px,median_abs_rotation_deg,median_abs_zoom,median_inliers,failed_pairs,ms_per_frame
0,tune,4402,2.081,9.250,0.008,0.002,899,0,23.805
1,val,2839,1.971,10.425,0.017,0.000,911,0,22.395
2,test,3763,2.944,19.533,0.018,0.002,743,15,17.213


In [ ]:
def same_object_iou(set_name):
    frames, gt_per_frame = frame_sets[set_name], experiment_utils.gt_frames(set_name, frame_sets[set_name])
    transforms = camera_motion[set_name][0]
    raw, compensated = [], []
    for _, video in frames.groupby('video', sort=False):
        index = video.index.values
        for previous, current in zip(index[:-1], index[1:]):
            boxes_prev, track_ids_prev = gt_per_frame[previous][0], gt_per_frame[previous][2]
            boxes_curr, track_ids_curr = gt_per_frame[current][0], gt_per_frame[current][2]
            common = np.intersect1d(track_ids_prev, track_ids_curr)
            if not len(common):
                continue
            prev = boxes_prev[[np.where(track_ids_prev == track)[0][0] for track in common]]
            curr = boxes_curr[[np.where(track_ids_curr == track)[0][0] for track in common]]
            raw += np.diag(temporal_postprocessing.iou_matrix(prev, curr)).tolist()
            warped = temporal_postprocessing.warp_boxes(prev, transforms[current])
            compensated += np.diag(temporal_postprocessing.iou_matrix(warped, curr)).tolist()
    return np.array(raw), np.array(compensated)

object_ious = {set_name: same_object_iou(set_name) for set_name in ['tune', 'val']}

In [ ]:
display(pd.DataFrame([dict(set=set_name, object_pairs=len(raw), median_iou_raw=np.median(raw),
                           median_iou_compensated=np.median(compensated), share_iou_below_0_3_raw=(raw < .3).mean(),
                           share_iou_below_0_3_compensated=(compensated < .3).mean())
                      for set_name, (raw, compensated) in object_ious.items()]).round(3))

,set,object_pairs,median_iou_raw,median_iou_compensated,share_iou_below_0_3_raw,share_iou_below_0_3_compensated
0,tune,214609,0.898,0.925,0.009,0.002
1,val,113284,0.873,0.895,0.042,0.008


In [ ]:
fig = go.Figure()
for set_name, (raw, compensated) in object_ious.items():
    fig.add_histogram(x=raw, name=f'{set_name}: raw', opacity=.55, nbinsx=50, histnorm='probability')
    fig.add_histogram(x=compensated, name=f'{set_name}: camera motion compensated', opacity=.55, nbinsx=50, histnorm='probability')
fig.update_layout(barmode='overlay', xaxis_title='IoU with its own box in the previous frame', yaxis_title='share of object pairs',
                  title='IoU of the same annotated object in consecutive frames, without and with camera-motion compensation')
save_plot(fig, 'camera_motion_iou')

In this analysis, we check how many missed objects could potentially be recovered using information from neighbouring frames. We consider an object missed if the model does not detect a box of the correct class with a score ≥ 0.3 and IoU ≥ 0.5.

For these objects, we check whether there is a lower-score prediction in the same frame that could be improved through rescoring, or whether the object was detected within +-2 neighbouring frames, which could allow us to use temporal propagation.

In [ ]:
tune_frames, tune_predictions = experiment_utils.load_preds(CACHE / 'tune_base.npz')
val_frames, val_predictions = experiment_utils.load_preds(CACHE / 'val_ens.npz')

In [ ]:
RECOVERY_KINDS = ['low_score_box_present', 'track_detected_nearby', 'low_score_and_track_nearby', 'no_box_but_track_nearby']

def recovery_row(label, set_name, frames, predictions):
    gt_per_frame = experiment_utils.gt_frames(set_name, frames)
    parts = []
    for _, video in frames.groupby('video', sort=False):
        index = video.index.values
        parts.append(temporal_postprocessing.recovery_potential([predictions[i] for i in index], [gt_per_frame[i] for i in index]))
    objects = sum(part['objects'] for part in parts)
    missed = sum(part['missed'] for part in parts)
    shares = {kind: sum(part[kind] * part['missed'] for part in parts) / missed for kind in RECOVERY_KINDS}
    return dict(set=label, objects=objects, missed_at_0_3=missed, missed_share=missed / objects, **shares)

In [ ]:
recovery = pd.DataFrame([recovery_row('tune: out-of-fold single checkpoint', 'tune', tune_frames, tune_predictions),
                         recovery_row('official val: 5-checkpoint ensemble', 'val', val_frames, val_predictions)])
recovery.to_csv(EXPERIMENT_DIR / 'recovery_potential.csv', index=False)
display(recovery.round(3))

,set,objects,missed_at_0_3,missed_share,low_score_box_present,track_detected_nearby,low_score_and_track_nearby,no_box_but_track_nearby
0,tune: out-of-fold single checkpoint,218305,64639,0.296,0.455,0.185,0.117,0.068
1,official val: 5-checkpoint ensemble,114132,41686,0.365,0.539,0.163,0.109,0.054


## Choosing the parameters

The search uses the 4,458 consecutive frames of the 56 training videos, each video predicted out-of-fold, and runs in three small stages instead of a full grid:
1. **Association** (`s_high`, `s_low`, `iou_link`, `max_gap`, `min_len`), with rescoring α = 0.5 and no suppression or gap filling.
2. **Rescoring strength α and isolated-box suppression β,** with the best association.
3. **Gap recovery γ.**


In [ ]:
TABLE_METRICS = ['mAP', 'AP50', 'AP75', 'APs', 'APm', 'APl', 'AR', 'ARs']
tune_groups = {'all': tune_frames.image_id.tolist(),
               **{f'fold{fold}': tune_frames.image_id[tune_frames.fold == fold].tolist() for fold in FOLDS}}
tuning_file = EXPERIMENT_DIR / 'tuning_results.json'
tuning = json.loads(tuning_file.read_text()) if tuning_file.exists() else {}

In [ ]:
def evaluate_settings(jobs):
    todo = [(key, params) for key, params in jobs if key not in tuning]
    if todo:
        results = experiment_utils.evaluate_many('tune', CACHE / 'tune_base.npz', todo, tune_groups)
        for key, (metrics, seconds, boxes) in results.items():
            tuning[key] = dict(params=dict(todo)[key], metrics=metrics, seconds_per_frame=seconds, boxes_per_frame=boxes)
        tuning_file.write_text(json.dumps(tuning, indent=1))
    rows = []
    for key, _ in jobs:
        entry = tuning[key]
        fold_gain = {f'dmAP_fold{f}': entry['metrics'][f'fold{f}']['mAP'] - tuning['base']['metrics'][f'fold{f}']['mAP'] for f in FOLDS}
        rows.append(dict(key=key, **(entry['params'] or {}), **{m: entry['metrics']['all'][m] for m in TABLE_METRICS},
                         **fold_gain, ms=1000 * entry['seconds_per_frame']))
    return pd.DataFrame(rows)

In [ ]:
evaluate_settings([('base', None)])
baseline = tuning['base']['metrics']['all']
print('out-of-fold single checkpoint, TTA + WBF, no temporal processing:', {m: round(v, 4) for m, v in baseline.items()})

out-of-fold single checkpoint, TTA + WBF, no temporal processing: {'mAP': 0.2614, 'AP50': 0.5099, 'AP75': 0.2431, 'APs': 0.1089, 'APm': 0.2319, 'APl': 0.4224, 'AR': 0.4299, 'ARs': 0.2723}


### Stage №1 association

In [ ]:
stage1_jobs = [(f"s1_h{s_high}_l{s_low}_i{iou_link}_g{max_gap}_m{min_len}",
                dict(s_high=s_high, s_low=s_low, iou_link=iou_link, max_gap=max_gap, min_len=min_len, alpha=.5, beta=1., gamma=0.))
               for s_high in [.3, .5] for s_low in [.01, .05] for iou_link in [.3, .5] for max_gap in [1, 2] for min_len in [3, 5]]
stage1 = evaluate_settings(stage1_jobs).sort_values('mAP', ascending=False)
stage1['dmAP'] = stage1.mAP - baseline['mAP']
display(stage1.head(8).round(4))
best_stage1 = tuning[stage1.key.iloc[0]]['params']

,key,s_high,s_low,iou_link,max_gap,min_len,alpha,beta,gamma,mAP,AP50,AP75,APs,APm,APl,AR,ARs,dmAP_fold0,dmAP_fold1,dmAP_fold2,dmAP_fold3,dmAP_fold4,ms,dmAP
6,s1_h0.3_l0.01_i0.5_g2_m3,0.3,0.01,0.5,2,3,0.5,1.0,0.0,0.2660,0.5187,0.2469,0.1114,0.2364,0.4288,0.43,0.2725,0.0088,0.0042,0.0036,0.0062,0.0027,5.3963,0.0046
7,s1_h0.3_l0.01_i0.5_g2_m5,0.3,0.01,0.5,2,5,0.5,1.0,0.0,0.2659,0.5187,0.2468,0.1114,0.2363,0.4285,0.43,0.2725,0.0087,0.0041,0.0035,0.0062,0.0027,5.4116,0.0046
4,s1_h0.3_l0.01_i0.5_g1_m3,0.3,0.01,0.5,1,3,0.5,1.0,0.0,0.2659,0.5186,0.2468,0.1114,0.2362,0.4287,0.43,0.2724,0.0085,0.0040,0.0035,0.0061,0.0027,5.6631,0.0045
5,s1_h0.3_l0.01_i0.5_g1_m5,0.3,0.01,0.5,1,5,0.5,1.0,0.0,0.2659,0.5185,0.2467,0.1114,0.2362,0.4284,0.43,0.2724,0.0085,0.0040,0.0034,0.0061,0.0027,5.6621,0.0045
2,s1_h0.3_l0.01_i0.3_g2_m3,0.3,0.01,0.3,2,3,0.5,1.0,0.0,0.2657,0.5184,0.2466,0.1118,0.2360,0.4285,0.43,0.2726,0.0088,0.0044,0.0032,0.0061,0.0024,5.8553,0.0044
3,s1_h0.3_l0.01_i0.3_g2_m5,0.3,0.01,0.3,2,5,0.5,1.0,0.0,0.2657,0.5184,0.2466,0.1118,0.2360,0.4283,0.43,0.2726,0.0088,0.0044,0.0032,0.0061,0.0024,5.9180,0.0043
0,s1_h0.3_l0.01_i0.3_g1_m3,0.3,0.01,0.3,1,3,0.5,1.0,0.0,0.2657,0.5183,0.2465,0.1118,0.2360,0.4284,0.43,0.2726,0.0088,0.0042,0.0030,0.0061,0.0024,5.7717,0.0043
1,s1_h0.3_l0.01_i0.3_g1_m5,0.3,0.01,0.3,1,5,0.5,1.0,0.0,0.2657,0.5182,0.2465,0.1118,0.2359,0.4281,0.43,0.2726,0.0087,0.0041,0.0030,0.0061,0.0024,5.7600,0.0043


### Stage №2 rescoring α and suppression β

In [ ]:
stage2_jobs = [(f"s2_a{alpha}_b{beta}", dict(best_stage1, alpha=alpha, beta=beta, gamma=0.))
               for alpha in [.2, .35, .5, .65, .8] for beta in [1., .8, .6, .4]]
stage2 = evaluate_settings(stage2_jobs).sort_values('mAP', ascending=False)
stage2['dmAP'] = stage2.mAP - baseline['mAP']
display(stage2.head(8).round(4))
best_stage2 = tuning[stage2.key.iloc[0]]['params']

,key,s_high,s_low,iou_link,max_gap,min_len,alpha,beta,gamma,mAP,AP50,AP75,APs,APm,APl,AR,ARs,dmAP_fold0,dmAP_fold1,dmAP_fold2,dmAP_fold3,dmAP_fold4,ms,dmAP
14,s2_a0.65_b0.6,0.3,0.01,0.5,2,3,0.65,0.6,0.0,0.2665,0.5203,0.2470,0.1127,0.2369,0.4306,0.4300,0.2724,0.0114,0.0044,0.0040,0.0071,0.0035,4.4147,0.0051
10,s2_a0.5_b0.6,0.3,0.01,0.5,2,3,0.50,0.6,0.0,0.2665,0.5197,0.2473,0.1126,0.2363,0.4294,0.4300,0.2725,0.0098,0.0041,0.0035,0.0067,0.0036,4.6317,0.0051
15,s2_a0.65_b0.4,0.3,0.01,0.5,2,3,0.65,0.4,0.0,0.2664,0.5202,0.2470,0.1126,0.2368,0.4305,0.4300,0.2724,0.0115,0.0042,0.0040,0.0071,0.0036,4.3150,0.0051
11,s2_a0.5_b0.4,0.3,0.01,0.5,2,3,0.50,0.4,0.0,0.2664,0.5196,0.2473,0.1126,0.2362,0.4293,0.4300,0.2725,0.0099,0.0039,0.0034,0.0067,0.0037,4.6305,0.0051
13,s2_a0.65_b0.8,0.3,0.01,0.5,2,3,0.65,0.8,0.0,0.2664,0.5202,0.2469,0.1123,0.2369,0.4306,0.4300,0.2725,0.0112,0.0046,0.0041,0.0070,0.0032,4.3975,0.0051
9,s2_a0.5_b0.8,0.3,0.01,0.5,2,3,0.50,0.8,0.0,0.2663,0.5195,0.2472,0.1121,0.2364,0.4293,0.4300,0.2725,0.0095,0.0043,0.0036,0.0065,0.0033,4.6170,0.0050
18,s2_a0.8_b0.6,0.3,0.01,0.5,2,3,0.80,0.6,0.0,0.2662,0.5198,0.2468,0.1126,0.2370,0.4318,0.4299,0.2724,0.0123,0.0047,0.0049,0.0066,0.0036,5.7550,0.0048
7,s2_a0.35_b0.4,0.3,0.01,0.5,2,3,0.35,0.4,0.0,0.2661,0.5186,0.2472,0.1124,0.2356,0.4282,0.4300,0.2725,0.0081,0.0035,0.0034,0.0047,0.0038,4.6318,0.0047


### Stage №3 gap recovery γ

In [ ]:
stage3_jobs = [(f"s3_c{gamma}", dict(best_stage2, gamma=gamma)) for gamma in [0., .25, .5, .75]]
stage3 = evaluate_settings(stage3_jobs).sort_values('mAP', ascending=False)
stage3['dmAP'] = stage3.mAP - baseline['mAP']
display(stage3.round(4))

,key,s_high,s_low,iou_link,max_gap,min_len,alpha,beta,gamma,mAP,AP50,AP75,APs,APm,APl,AR,ARs,dmAP_fold0,dmAP_fold1,dmAP_fold2,dmAP_fold3,dmAP_fold4,ms,dmAP
3,s3_c0.75,0.3,0.01,0.5,2,3,0.65,0.6,0.75,0.2666,0.5204,0.247,0.1127,0.2369,0.4306,0.4301,0.2726,0.0115,0.0045,0.004,0.0071,0.0036,3.9346,0.0052
2,s3_c0.5,0.3,0.01,0.5,2,3,0.65,0.6,0.50,0.2666,0.5204,0.247,0.1127,0.2369,0.4306,0.4301,0.2726,0.0115,0.0045,0.004,0.0071,0.0036,3.9396,0.0052
1,s3_c0.25,0.3,0.01,0.5,2,3,0.65,0.6,0.25,0.2666,0.5204,0.247,0.1127,0.2369,0.4306,0.4301,0.2725,0.0115,0.0045,0.004,0.0071,0.0036,3.9309,0.0052
0,s3_c0.0,0.3,0.01,0.5,2,3,0.65,0.6,0.00,0.2665,0.5203,0.247,0.1127,0.2369,0.4306,0.4300,0.2724,0.0114,0.0044,0.004,0.0071,0.0035,3.8672,0.0051


In [ ]:
FINAL = tuning[stage3.key.iloc[0]]['params']
frozen = dict(params=FINAL, chosen_on='out-of-fold consecutive frames of the 56 training videos',
              selection='highest mAP@[.50:.95] over all tuning frames')
(EXPERIMENT_DIR / 'temporal_config.json').write_text(json.dumps(frozen, indent=2))
print('frozen parameters:', FINAL)

frozen parameters: {'s_high': 0.3, 's_low': 0.01, 'iou_link': 0.5, 'max_gap': 2, 'min_len': 3, 'alpha': 0.65, 'beta': 0.6, 'gamma': 0.75}


In [ ]:
fig = make_subplots(1, 3, subplot_titles=['stage 1: association (top 12)', 'stage 2: α and β', 'stage 3: gap recovery γ'])
top12 = stage1.head(12)
fig.add_bar(x=top12.key.str.replace('s1_', ''), y=top12.dmAP, marker_color='#4c78a8', showlegend=False, row=1, col=1)
for beta in sorted(stage2.beta.unique()):
    rows = stage2[stage2.beta == beta].sort_values('alpha')
    fig.add_scatter(x=rows.alpha, y=rows.dmAP, mode='lines+markers', name=f'β = {beta}', row=1, col=2)
rows = stage3.sort_values('gamma')
fig.add_scatter(x=rows.gamma, y=rows.dmAP, mode='lines+markers', name='γ', line_color='#e45756', row=1, col=3)
fig.update_yaxes(title_text='ΔmAP vs no temporal processing', row=1, col=1)
fig.update_xaxes(title_text='α', row=1, col=2)
fig.update_xaxes(title_text='γ', row=1, col=3)
fig.update_layout(title='Parameter search on held-out training videos (out-of-fold, single checkpoint, TTA + WBF)', height=520)
save_plot(fig, 'parameter_search')

## Ablation (tuning frames)

Each component is evaluated alone and in combination, with the frozen association parameters:
* camera-motion compensation off;
* rescoring only;
* suppression only;
* gap recovery only;
* the full method.

In [ ]:
ABLATION = {'no temporal processing': None,
            'rescoring only': dict(FINAL, beta=1., gamma=0.),
            'suppression only': dict(FINAL, alpha=0., gamma=0.),
            'gap recovery only': dict(FINAL, alpha=0., beta=1.),
            'rescoring + suppression': dict(FINAL, gamma=0.),
            'full method': FINAL,
            'full method without camera motion': dict(FINAL, motion=False)}

In [ ]:
ablation_tune = evaluate_settings([(f'abl_{name}', params) for name, params in ABLATION.items()])
ablation_tune.insert(0, 'variant', list(ABLATION))
ablation_tune['dmAP'] = ablation_tune.mAP - baseline['mAP']
ablation_tune['dAPs'] = ablation_tune.APs - baseline['APs']
columns = ['variant', 'mAP', 'dmAP', 'AP50', 'AP75', 'APs', 'dAPs', 'AR', 'ARs'] + [f'dmAP_fold{f}' for f in FOLDS] + ['ms']
display(ablation_tune[columns].round(4))
ablation_tune.to_csv(EXPERIMENT_DIR / 'ablation_tuning_frames.csv', index=False)

,variant,mAP,dmAP,AP50,AP75,APs,dAPs,AR,ARs,dmAP_fold0,dmAP_fold1,dmAP_fold2,dmAP_fold3,dmAP_fold4,ms
0,no temporal processing,0.2614,0.0000,0.5099,0.2431,0.1089,0.0000,0.4299,0.2723,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,rescoring only,0.2661,0.0047,0.5195,0.2466,0.1114,0.0025,0.4300,0.2725,0.0105,0.0045,0.0042,0.0067,0.0027,5.4729
2,suppression only,0.2623,0.0010,0.5118,0.2440,0.1102,0.0013,0.4299,0.2724,0.0016,0.0006,0.0003,0.0007,0.0011,5.2803
3,gap recovery only,0.2614,0.0000,0.5100,0.2431,0.1090,0.0001,0.4300,0.2725,0.0001,0.0000,0.0000,0.0000,0.0001,5.5882
4,rescoring + suppression,0.2665,0.0051,0.5203,0.2470,0.1127,0.0037,0.4300,0.2724,0.0114,0.0044,0.0040,0.0071,0.0035,5.3499
5,full method,0.2666,0.0052,0.5204,0.2470,0.1127,0.0038,0.4301,0.2726,0.0115,0.0045,0.0040,0.0071,0.0036,5.3188
6,full method without camera motion,0.2660,0.0047,0.5202,0.2463,0.1124,0.0034,0.4301,0.2726,0.0111,0.0047,0.0029,0.0072,0.0034,5.3287


## Held-out evaluation on official validation

In [ ]:
val_groups = {'all': val_frames.image_id.tolist(),
              **{video: frames.image_id.tolist() for video, frames in val_frames.groupby('video', sort=False)},
              **{f'short side {h}': val_frames.image_id[val_frames.height == h].tolist() for h in sorted(val_frames.height.unique())}}
val_results_file = EXPERIMENT_DIR / 'official_validation_raw.json'
val_results = {}
if val_results_file.exists():
    val_results = {key: tuple(value) for key, value in json.loads(val_results_file.read_text()).items()}

In [ ]:
for predictions_name in ['val_ens'] + [f'val_fold{fold}' for fold in FOLDS]:
    if predictions_name == 'val_ens':
        jobs = [(f'{predictions_name}|{name}', params) for name, params in ABLATION.items()]
    else:
        jobs = [(f'{predictions_name}|no temporal processing', None), (f'{predictions_name}|full method', FINAL)]
    todo = [(key, params) for key, params in jobs if key not in val_results]
    if todo:
        val_results.update(experiment_utils.evaluate_many('val', CACHE / f'{predictions_name}.npz', todo, val_groups))
        val_results_file.write_text(json.dumps(val_results, indent=1))

In [ ]:
rows = []
for key, (metrics, seconds, boxes) in val_results.items():
    predictions_name, variant = key.split('|')
    rows.append(dict(predictions=predictions_name, variant=variant, **{m: metrics['all'][m] for m in TABLE_METRICS},
                     boxes_per_frame=boxes, ms_per_frame=1000 * seconds))
val_table = pd.DataFrame(rows)
without_temporal = val_table[val_table.variant == 'no temporal processing'].set_index('predictions')
for metric in ['mAP', 'APs', 'AR']:
    val_table[f'd{metric}'] = val_table[metric] - val_table.predictions.map(without_temporal[metric])
val_table.to_csv(EXPERIMENT_DIR / 'official_validation_results.csv', index=False)

In [ ]:
ensemble_rows = val_table[val_table.predictions == 'val_ens']
display(ensemble_rows[['variant', 'mAP', 'dmAP', 'AP50', 'AP75', 'APs', 'dAPs', 'AR', 'dAR', 'boxes_per_frame', 'ms_per_frame']].round(4))
single_checkpoints = val_table[(val_table.predictions != 'val_ens') & (val_table.variant == 'full method')]
display(single_checkpoints[['predictions', 'mAP', 'dmAP', 'APs', 'dAPs', 'AR', 'dAR']].round(4))

,variant,mAP,dmAP,AP50,AP75,APs,dAPs,AR,dAR,boxes_per_frame,ms_per_frame
0,no temporal processing,0.2364,0.0000,0.4771,0.2068,0.1267,0.0000,0.4727,0.0000,451.6841,0.0000
1,rescoring only,0.2447,0.0083,0.4983,0.2120,0.1315,0.0048,0.4728,0.0001,451.6841,3.1900
2,suppression only,0.2376,0.0012,0.4801,0.2078,0.1272,0.0005,0.4727,0.0001,451.6841,3.2011
3,gap recovery only,0.2364,0.0000,0.4772,0.2069,0.1267,0.0000,0.4729,0.0002,451.7059,3.2623
4,rescoring + suppression,0.2449,0.0085,0.4986,0.2123,0.1314,0.0048,0.4728,0.0001,451.6841,3.2190
5,full method,0.2450,0.0086,0.4987,0.2124,0.1315,0.0048,0.4731,0.0004,451.7059,3.2650
6,full method without camera motion,0.2442,0.0078,0.4953,0.2128,0.1276,0.0010,0.4728,0.0001,451.7119,4.1427


,predictions,mAP,dmAP,APs,dAPs,AR,dAR
8,val_fold0,0.2281,0.0070,0.1243,0.0082,0.3970,0.0005
10,val_fold1,0.2262,0.0122,0.1236,0.0046,0.3667,0.0007
12,val_fold2,0.2357,0.0105,0.1189,-0.0000,0.4364,0.0007
14,val_fold3,0.2170,0.0057,0.1092,0.0034,0.3820,0.0009
16,val_fold4,0.2279,0.0076,0.1174,0.0077,0.3822,0.0008


In [ ]:
variants = ['no temporal processing', 'full method']
per_group = pd.DataFrame({group: {v: val_results[f'val_ens|{v}'][0][group]['mAP'] for v in variants} for group in val_groups}).T
per_group['dmAP'] = per_group['full method'] - per_group['no temporal processing']
per_group['frames'] = [len(val_groups[group]) for group in per_group.index]
display(per_group.round(4))

,no temporal processing,full method,dmAP,frames
all,0.2364,0.2450,0.0086,2846
uav0000086_00000_v,0.1610,0.1787,0.0177,464
uav0000117_02622_v,0.2915,0.3037,0.0122,349
uav0000137_00458_v,0.3066,0.3156,0.0089,233
uav0000182_00000_v,0.2037,0.2144,0.0107,363
uav0000268_05773_v,0.1810,0.1844,0.0035,978
uav0000305_00000_v,0.3194,0.3258,0.0064,184
uav0000339_00001_v,0.2785,0.2969,0.0184,275
short side 756,0.2443,0.2609,0.0167,827
short side 1071,0.2803,0.2968,0.0165,459


In [ ]:
ensemble_gain = float(ensemble_rows[ensemble_rows.variant == 'full method'].dmAP.iloc[0])
singles_improved = int((single_checkpoints.dmAP > 0).sum())
SUBMIT = ensemble_gain >= 0.002 and singles_improved >= 4
print(f'5-checkpoint ensemble: ΔmAP {ensemble_gain:+.4f} | single checkpoints improved: {singles_improved}/5 | decision rule met: {SUBMIT}')

5-checkpoint ensemble: ΔmAP +0.0086 | single checkpoints improved: 5/5 | decision rule met: True


In [ ]:
val_transforms = camera_motion['val'][0]
metrics_500 = {}
for name, params in [('no temporal processing', None), ('full method', FINAL)]:
    if params is None:
        predictions = val_predictions
    else:
        predictions = experiment_utils.apply_temporal(val_frames, val_predictions, val_transforms, params)
    metrics_500[name] = temporal_postprocessing.coco_metrics(val_gt, val_frames.image_id.tolist(), predictions, max_dets=500)['all']
display(pd.DataFrame(metrics_500).T.round(4))

,mAP,AP50,AP75,APs,APm,APl,AR,ARs
no temporal processing,0.2368,0.4783,0.2071,0.1268,0.2228,0.3998,0.4753,0.3004
full method,0.2454,0.4999,0.2125,0.1316,0.2283,0.4221,0.4756,0.3004


## Errors, localisation and failure cases

In [ ]:
val_temporal = experiment_utils.apply_temporal(val_frames, val_predictions, val_transforms, FINAL)
annotations_by_image = {}
for annotation in experiment_utils.annotations_for('val')['annotations']:
    annotations_by_image.setdefault(int(annotation['image_id']), []).append(annotation)

def xyxy(annotation):
    x, y, w, h = annotation['bbox']
    return [x, y, x + w, y + h]

In [ ]:
def count_errors(prediction, frame_annotations, score_threshold=.3):
    objects = [ann for ann in frame_annotations if not ann.get('iscrowd', 0)]
    gt_boxes = np.array([xyxy(ann) for ann in objects]).reshape(-1, 4)
    gt_labels = np.array([ann['category_id'] for ann in objects])
    ignore = np.array([xyxy(ann) for ann in frame_annotations if ann.get('iscrowd', 0)]).reshape(-1, 4)
    boxes, scores, labels = prediction
    keep = scores >= score_threshold
    boxes, labels = boxes[keep].astype(np.float64), labels[keep]
    iou = temporal_postprocessing.iou_matrix(boxes, gt_boxes)
    used = np.zeros(len(gt_boxes), bool)
    found, false_positives, matched_ious = 0, 0, []
    for j in range(len(boxes)):
        candidates = (iou[j] >= .5) & ~used & (gt_labels == labels[j])
        if candidates.any():
            best = np.where(candidates)[0][iou[j][candidates].argmax()]
            used[best] = True
            found += 1
            matched_ious.append(iou[j, best])
        elif len(ignore):
            top_left = np.maximum(boxes[j, :2], ignore[:, :2])
            bottom_right = np.minimum(boxes[j, 2:], ignore[:, 2:])
            share_in_ignore = np.clip(bottom_right - top_left, 0, None).prod(1) / max(np.prod(boxes[j, 2:] - boxes[j, :2]), 1e-6)
            false_positives += int(share_in_ignore.max() < .5)
        else:
            false_positives += 1
    return found, false_positives, len(gt_boxes) - found, matched_ious

In [ ]:
rows = []
for i, image_id in enumerate(val_frames.image_id):
    frame_annotations = annotations_by_image.get(int(image_id), [])
    before = count_errors(val_predictions[i], frame_annotations)
    after = count_errors(val_temporal[i], frame_annotations)
    rows.append(dict(image_id=image_id, video=val_frames.video[i], frame=val_frames.frame[i],
                     tp_before=before[0], fp_before=before[1], fn_before=before[2],
                     tp_after=after[0], fp_after=after[1], fn_after=after[2],
                     iou_before=np.mean(before[3]) if before[3] else np.nan, iou_after=np.mean(after[3]) if after[3] else np.nan))
per_frame = pd.DataFrame(rows)
per_frame.to_csv(EXPERIMENT_DIR / 'val_per_frame_errors.csv', index=False)

In [ ]:
error_summary = pd.DataFrame({
    'before': [per_frame.tp_before.sum(), per_frame.fp_before.sum(), per_frame.fn_before.sum(), per_frame.iou_before.mean()],
    'after (full method)': [per_frame.tp_after.sum(), per_frame.fp_after.sum(), per_frame.fn_after.sum(), per_frame.iou_after.mean()]},
    index=['found (TP)', 'false positives', 'missed (FN)', 'mean IoU of matched boxes'])
error_summary['change'] = error_summary['after (full method)'] - error_summary['before']
display(error_summary.round(4))

,before,after (full method),change
found (TP),71329.0000,74070.0000,2741.0000
false positives,41765.0000,43848.0000,2083.0000
missed (FN),42803.0000,40062.0000,-2741.0000
mean IoU of matched boxes,0.7884,0.7837,-0.0047


### Best and worst frame

In [ ]:
def outline(boxes, color, hover_texts=None):
    xs, ys, texts = [], [], []
    for k, (x1, y1, x2, y2) in enumerate(boxes):
        xs += [x1, x2, x2, x1, x1, None]
        ys += [y1, y1, y2, y2, y1, None]
        texts += [hover_texts[k] if hover_texts else ''] * 5 + [None]
    return go.Scatter(x=xs, y=ys, mode='lines', line=dict(color=color, width=1.1), text=texts, hoverinfo='text', showlegend=False)

def image_as_data_url(path):
    image = Image.open(path).convert('RGB')
    width, height = image.size
    image.thumbnail((1280, 1280))
    buffer = io.BytesIO()
    image.save(buffer, 'JPEG', quality=85)
    return 'data:image/jpeg;base64,' + base64.b64encode(buffer.getvalue()).decode(), width, height

In [ ]:
def frame_figure(i, title):
    path = val_frames.path[i] if 'path' in val_frames else experiment_utils.frames_meta('val').path[i]
    source, width, height = image_as_data_url(path)
    frame_annotations = annotations_by_image.get(int(val_frames.image_id[i]), [])
    gt_boxes = [xyxy(ann) for ann in frame_annotations if not ann.get('iscrowd', 0)]
    fig = make_subplots(1, 2, horizontal_spacing=.01, subplot_titles=['5-checkpoint ensemble (submitted)', 'with temporal post-processing'])
    for column, (boxes, scores, labels) in enumerate([val_predictions[i], val_temporal[i]], 1):
        keep = scores >= .3
        names = [f'{tta_wbf.KAGGLE_CLASSES[int(label) - 1]} {score:.2f}' for label, score in zip(labels[keep], scores[keep])]
        fig.add_trace(outline(gt_boxes, 'lime'), 1, column)
        fig.add_trace(outline(boxes[keep].tolist(), 'red', names), 1, column)
        fig.add_layout_image(source=source, xref=f'x{column}', yref=f'y{column}', x=0, y=0, sizex=width, sizey=height,
                             xanchor='left', yanchor='top', sizing='stretch', layer='below')
    fig.update_xaxes(range=[0, width], visible=False)
    fig.update_yaxes(range=[height, 0], visible=False)
    fig.update_layout(title=title + ' · green = ground truth, red = prediction with score ≥ 0.3', width=1600,
                      height=int(560 * height / width) + 120, margin=dict(l=10, r=10, t=90, b=10), template='plotly_white')
    return fig

In [ ]:
net_gain = (per_frame.tp_after - per_frame.fp_after) - (per_frame.tp_before - per_frame.fp_before)
cases = [('best_frame', int(net_gain.idxmax()), 'largest gain (found − false positives)'),
         ('worst_frame', int(net_gain.idxmin()), 'failure case: largest loss (found − false positives)')]
VISUALIZATIONS_DIR = EXPERIMENT_DIR / 'visualizations'
VISUALIZATIONS_DIR.mkdir(exist_ok=True)
for label, i, reason in cases:
    row = per_frame.loc[i]
    counts = f"TP {row.tp_before}→{row.tp_after}, FP {row.fp_before}→{row.fp_after}, FN {row.fn_before}→{row.fn_after}"
    fig = frame_figure(i, f"{row.video} frame {row.frame}: {reason}<br>{counts}")
    fig.write_html(VISUALIZATIONS_DIR / f'{label}_{row.video}_{row.frame}.html', include_plotlyjs='directory')
    print(label, row.video, row.frame, f'TP {row.tp_before}->{row.tp_after}, FP {row.fp_before}->{row.fp_after}')

best_frame uav0000137_00458_v 66 TP 81->84, FP 44->36


worst_frame uav0000137_00458_v 171 TP 78->79, FP 27->40


### Gain per video and ablation on official validation

In [ ]:
groups = per_group.drop(index='all')
colors = ['#54a24b' if value > 0 else '#e45756' for value in groups.dmAP]
labels = [f'{value:+.4f} ({frames} frames)' for value, frames in zip(groups.dmAP, groups.frames)]
fig = go.Figure(go.Bar(x=groups.index, y=groups.dmAP, marker_color=colors, text=labels))
fig.update_layout(title='Official validation: mAP change from temporal post-processing per video and per frame resolution (5-checkpoint ensemble)',
                  yaxis_title='ΔmAP@[.50:.95]')
save_plot(fig, 'val_delta_per_video')

In [ ]:
fig = make_subplots(1, 2, subplot_titles=['ΔmAP vs submitted ensemble', 'ΔAP_small'])
fig.add_bar(x=ensemble_rows.variant, y=ensemble_rows.dmAP, marker_color='#4c78a8', showlegend=False, row=1, col=1)
fig.add_bar(x=ensemble_rows.variant, y=ensemble_rows.dAPs, marker_color='#f58518', showlegend=False, row=1, col=2)
fig.update_layout(title='Ablation on official validation (frozen parameters, 5-checkpoint ensemble)', height=520)
save_plot(fig, 'ablation_official_validation')

## Latency


In [ ]:
tracking_ms = float(ensemble_rows[ensemble_rows.variant == 'full method'].ms_per_frame.iloc[0])
latency = pd.DataFrame([dict(step='camera motion (incl. grayscale decoding)', set=set_name, ms_per_frame=1000 * camera_motion[set_name][2].mean())
                        for set_name in frame_sets] +
                       [dict(step='tracking + rescoring + gap recovery', set='val (5-checkpoint ensemble)', ms_per_frame=tracking_ms)])
display(latency.round(1))
latency.to_csv(EXPERIMENT_DIR / 'latency.csv', index=False)

,step,set,ms_per_frame
0,camera motion (incl. grayscale decoding),tune,23.8
1,camera motion (incl. grayscale decoding),val,22.4
2,camera motion (incl. grayscale decoding),test,17.2
3,tracking + rescoring + gap recovery,val (5-checkpoint ensemble),3.3


In [ ]:
if SUBMIT:
    test_temporal = experiment_utils.apply_temporal(test_frames, test_predictions, camera_motion['test'][0], FINAL)
    submission_file = experiment_utils.ROOT / 'kaggle_submissions' / 'faster_rcnn_submission3.csv'
    strings = [tta_wbf.prediction_string(boxes, scores, labels, height, width)
               for (boxes, scores, labels), height, width in zip(test_temporal, test_frames.height, test_frames.width)]
    pd.DataFrame({'image_id': test_frames.image_id, 'PredictionString': strings}).to_csv(submission_file, index=False)
    checks, stats = tta_wbf.validate_submission(submission_file, pd.read_csv(experiment_utils.DATA / 'test_images.csv'))
    for check, passed in checks.items():
        print('PASS' if passed else 'FAIL', check)
    print(stats)
    assert all(checks.values())
    print('saved', submission_file)
else:
    print('The decision rule was not met: no submission is created.')

PASS columns are exactly image_id, PredictionString
PASS one row per test image, same order as test_images.csv
PASS no duplicate image ids
PASS no missing image ids
PASS every prediction is "class score x y w h" with class 0-7, score in (0, 1], finite box inside the image
PASS at most 500 detections per image
PASS empty PredictionString is an empty field (no NaN/None text)
{'rows': 3811, 'images_with_detections': 3811, 'images_without_detections': 0, 'detections': 1783078, 'max_per_image': 500, 'mean_per_image': 467.87667278929416, 'classes_present': [0, 1, 2, 3, 4, 5, 6, 7], 'min_score': 0.0002, 'first_errors': []}
saved E:\ucu_hw2\kaggle_submissions\faster_rcnn_submission3.csv


## The same method on every detector (official validation)

The temporal post-processing developed above is applied unchanged to the 5-checkpoint TTA + WBF ensemble of every detector;
its parameters were tuned separately for each detector on its out-of-fold tuning frames.

In [ ]:
ROOT = experiment_utils.ROOT
detectors = [
    ('Faster R-CNN v1', 'fcos/inference/official_validation_results.csv',
     'Faster R-CNN 5 checkpoints TTA (0.2587 pipeline)', 'Faster R-CNN + temporal (0.2810 pipeline)'),
    ('Faster R-CNN v2 + scale aug', 'faster_rcnn/faster_rcnn_v2/inference/official_validation_results.csv',
     'v2 5 checkpoints TTA', 'A: v2 + temporal (tuned)'),
    ('FCOS', 'fcos/inference/official_validation_results.csv',
     'FCOS 5 checkpoints, optimized TTA', 'FCOS 5 checkpoints, optimized TTA + temporal (tuned for FCOS)'),
    ('RetinaNet', 'retinanet/inference/official_validation_results.csv',
     'RetinaNet 5 checkpoints TTA + WBF', 'RetinaNet 5-fold TTA + WBF + temporal (tuned)'),
    ('RT-DETR', 'rt_detr/inference/official_validation_results.csv',
     'RT-DETR 5 checkpoints TTA + WBF', 'RT-DETR 5-fold TTA + WBF + temporal (tuned)'),
]
rows = []
for detector, csv_file, before_key, after_key in detectors:
    table = pd.read_csv(ROOT / csv_file).set_index('key')
    before, after = table.loc[before_key], table.loc[after_key]
    videos = [column for column in table.columns if column.startswith('mAP_video_')]
    rows.append(dict(detector=detector, mAP_without=before['mAP'], mAP_with=after['mAP'], gain=after['mAP'] - before['mAP'],
                     AP_small_gain=after['APs'] - before['APs'],
                     videos_improved=f"{int((after[videos] > before[videos]).sum())} / {len(videos)}"))
print(pd.DataFrame(rows).round(4).to_string())

                      detector  mAP_without  mAP_with    gain  AP_small_gain videos_improved
0              Faster R-CNN v1       0.2364    0.2450  0.0086         0.0048           7 / 7
1  Faster R-CNN v2 + scale aug       0.2717    0.2841  0.0124         0.0225           7 / 7
2                         FCOS       0.2216    0.2367  0.0150         0.0134           7 / 7
3                    RetinaNet       0.2249    0.2419  0.0170         0.0009           6 / 7
4                      RT-DETR       0.2882    0.3012  0.0129         0.0036           7 / 7
